# 00_01: Tokenization and Embeddings

**Learning Objectives:**
- Understand why tokenization is needed and how it works
- Compare tokenization algorithms: BPE, WordPiece, SentencePiece, and Unigram
- Explore vocabularies, special tokens, and token IDs
- Understand token embeddings and positional encodings
- Use HuggingFace tokenizers to inspect real model tokenization

**Prerequisites:** Basic Python. No ML experience required — this is the starting point.

## Prerequisites

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 4GB minimum |
| **Storage** | 1GB free |
| **GPU** | Not required |

## Expected Behaviors

### What You'll See
- Tokenizers split text into subword pieces (not just words)
- Different models tokenize the same text differently
- Rare words get split into smaller pieces; common words stay whole
- Each token maps to an integer ID and then to an embedding vector

### First Run
- Tokenizer downloads are small (~1MB each), very fast
- Model downloads for embedding inspection: ~250-500MB

### Common Observations
- "unhappiness" might tokenize as ["un", "happiness"] or ["un", "happ", "iness"]
- GPT-2 and BERT tokenize the same sentence very differently
- Vocabulary sizes range from 30k (BERT) to 50k+ (GPT-2)

## Overview

Before a neural network can process text, the text must be converted into numbers. This conversion happens in two stages:

1. **Tokenization**: Split text into meaningful pieces (tokens)
2. **Embedding**: Map each token to a dense vector representation

This notebook explores both stages. Understanding tokenization is critical because it affects everything downstream — model performance, context window limits, and even how much you pay for API calls (which are priced per token).

### Why Not Just Split on Spaces?

Word-level tokenization has serious problems:
- **Huge vocabulary**: English has 170,000+ words, plus names, typos, code...
- **Out-of-vocabulary (OOV)**: Any word not in the vocabulary becomes `[UNK]`
- **No morphology**: "run", "running", "runner" are treated as completely unrelated

Modern tokenizers solve this with **subword tokenization** — a middle ground between character-level and word-level that handles any input while keeping vocabulary manageable.

## Setup and Installation

In [ ]:
# Import libraries
import random
import sys
import numpy as np
import torch
import transformers
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Why Tokenization Matters

Let's start by seeing the problem firsthand. We'll compare three approaches to splitting text: character-level, word-level, and subword-level. This makes the motivation for subword tokenization concrete.

In [ ]:
# Three approaches to tokenization
sample_text = "The unhappiest programmer couldn't debug the TypeError."

# Approach 1: Character-level
char_tokens = list(sample_text)

# Approach 2: Word-level (naive split on spaces + punctuation)
import re
word_tokens = re.findall(r"\w+|[^\w\s]", sample_text)

# Approach 3: Subword (using a real tokenizer)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
subword_tokens = tokenizer.tokenize(sample_text.lower())

import pandas as pd

print(f"Input: \"{sample_text}\"\n")
comparison = pd.DataFrame({
    "Method": ["Character", "Word", "Subword (BERT)"],
    "Tokens": [str(char_tokens[:15]) + "...", str(word_tokens), str(subword_tokens)],
    "Count": [len(char_tokens), len(word_tokens), len(subword_tokens)],
})
print(comparison.to_string(index=False))

print(f"\nNotice how BERT splits 'unhappiest' into subwords: {tokenizer.tokenize('unhappiest')}")
print(f"And 'TypeError' becomes: {tokenizer.tokenize('typeerror')}")

### The Vocabulary Trade-off

There's a fundamental trade-off between vocabulary size and sequence length. Character tokenization uses a tiny vocabulary (~100 tokens) but produces very long sequences. Word tokenization keeps sequences short but needs an enormous vocabulary. Subword tokenization hits the sweet spot.

In [ ]:
# Vocabulary size comparison
vocab_df = pd.DataFrame({
    "Method": ["Character", "Subword (BPE)", "Subword (WordPiece)", "Word-level"],
    "Typical Vocab Size": ["~100-300", "30,000-50,000", "30,000", "100,000+"],
    "Handles Unknown Words?": ["Yes (any char)", "Yes (falls back to chars)", "Yes (falls back to chars)", "No ([UNK] token)"],
    "Sequence Length": ["Very long", "Moderate", "Moderate", "Short"],
    "Used By": ["ByT5", "GPT-2, Llama, Mistral", "BERT, DistilBERT", "(Legacy systems)"],
})
print("=== TOKENIZATION METHODS ===")
print(vocab_df.to_string(index=False))

## Part 2: Tokenization Algorithms

The three dominant subword algorithms are **Byte-Pair Encoding (BPE)**, **WordPiece**, and **Unigram**. They all produce subword tokens, but they differ in how they build the vocabulary. Let's see each in action using real HuggingFace tokenizers.

### 2.1 Byte-Pair Encoding (BPE)

BPE starts with individual characters and iteratively merges the most frequent pair. After training, it applies learned merge rules to new text. GPT-2, Llama, and most modern LLMs use BPE.

**Algorithm:**
1. Start with character-level vocabulary
2. Count all adjacent character pairs
3. Merge the most frequent pair into a new token
4. Repeat until vocabulary reaches target size

In [ ]:
# BPE tokenizer: GPT-2
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

test_sentences = [
    "Hello, how are you?",
    "The unhappiest programmer debugged the TypeError.",
    "Pneumonoultramicroscopicsilicovolcanoconiosis is a lung disease.",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
]

print("=== GPT-2 (BPE) TOKENIZATION ===")
print(f"Vocabulary size: {gpt2_tokenizer.vocab_size:,}\n")

for sentence in test_sentences:
    tokens = gpt2_tokenizer.tokenize(sentence)
    ids = gpt2_tokenizer.encode(sentence)
    print(f"Input:  \"{sentence}\"")
    print(f"Tokens: {tokens}")
    print(f"IDs:    {ids}")
    print(f"Count:  {len(tokens)}")
    print()

### 2.2 WordPiece

WordPiece is similar to BPE but uses a different merge criterion — it maximizes the likelihood of the training data rather than just counting pair frequency. BERT and DistilBERT use WordPiece. Subword continuations are marked with `##` prefix.

In [ ]:
# WordPiece tokenizer: BERT
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("=== BERT (WordPiece) TOKENIZATION ===")
print(f"Vocabulary size: {bert_tokenizer.vocab_size:,}\n")

for sentence in test_sentences:
    tokens = bert_tokenizer.tokenize(sentence)
    ids = bert_tokenizer.encode(sentence)
    print(f"Input:  \"{sentence}\"")
    print(f"Tokens: {tokens}")
    print(f"IDs:    {ids[:15]}{'...' if len(ids) > 15 else ''}")
    print(f"Count:  {len(tokens)}")
    print()

print("Notice the ## prefix on continuation tokens (e.g., 'un' + '##happi' + '##est')")

### 2.3 Side-by-Side Comparison

Different tokenizers produce different token counts for the same input. This matters because transformer models have fixed context windows measured in tokens, and API pricing is per-token. Let's compare how GPT-2 and BERT handle the same text.

In [ ]:
# Side-by-side comparison
comparison_text = "Transformers revolutionized natural language processing in 2017."

tokenizers = {
    "GPT-2 (BPE)": gpt2_tokenizer,
    "BERT (WordPiece)": bert_tokenizer,
}

print(f"Input: \"{comparison_text}\"\n")
print(f"{'Tokenizer':<22} {'Token Count':<13} {'Tokens'}")
print("-" * 80)

for name, tok in tokenizers.items():
    tokens = tok.tokenize(comparison_text)
    print(f"{name:<22} {len(tokens):<13} {tokens}")

## Part 3: Special Tokens

Every tokenizer defines special tokens that serve structural purposes. These tokens tell the model where sentences begin and end, separate segments, represent unknown words, and handle padding. Different model families use different special tokens.

In [ ]:
# Compare special tokens across tokenizers
print("=== SPECIAL TOKENS ===")

for name, tok in tokenizers.items():
    print(f"\n{name}:")
    print(f"  [PAD] = {tok.pad_token!r} (ID: {tok.pad_token_id})")
    print(f"  [UNK] = {tok.unk_token!r} (ID: {tok.unk_token_id})")
    print(f"  [BOS] = {tok.bos_token!r} (ID: {tok.bos_token_id})")
    print(f"  [EOS] = {tok.eos_token!r} (ID: {tok.eos_token_id})")
    print(f"  [SEP] = {tok.sep_token!r} (ID: {getattr(tok, 'sep_token_id', None)})")
    print(f"  [CLS] = {tok.cls_token!r} (ID: {getattr(tok, 'cls_token_id', None)})")

In [ ]:
# See how special tokens are added to encoded input
text = "Hello world"

print("=== ENCODING WITH SPECIAL TOKENS ===")
for name, tok in tokenizers.items():
    encoded = tok(text, return_tensors=None)
    tokens = tok.convert_ids_to_tokens(encoded['input_ids'])
    print(f"\n{name}:")
    print(f"  Tokens:        {tokens}")
    print(f"  IDs:           {encoded['input_ids']}")
    print(f"  Attention Mask: {encoded['attention_mask']}")

print("\nBERT wraps with [CLS]...[SEP]. GPT-2 adds no special tokens by default.")

## Part 4: Exploring Vocabularies

Each tokenizer has a fixed vocabulary mapping tokens to integer IDs. Let's explore what's inside these vocabularies — you'll find whole words, subword fragments, punctuation, and sometimes surprising entries.

In [ ]:
# Explore the GPT-2 vocabulary
vocab = gpt2_tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])

print(f"=== GPT-2 VOCABULARY ===")
print(f"Total tokens: {len(vocab):,}")
print(f"\nFirst 20 tokens (lowest IDs):")
for token, idx in sorted_vocab[:20]:
    print(f"  {idx:>5}: {token!r}")

print(f"\nLast 10 tokens (highest IDs):")
for token, idx in sorted_vocab[-10:]:
    print(f"  {idx:>5}: {token!r}")

# Search for specific patterns
print(f"\nTokens containing 'python': ", end="")
python_tokens = [t for t in vocab if 'python' in t.lower()]
print(python_tokens[:10])

In [ ]:
# Token ID lookup: encode and decode
def inspect_tokenization(text: str, tokenizer: AutoTokenizer) -> None:
    """Show detailed tokenization breakdown for a text string.

    Args:
        text: Input text to tokenize.
        tokenizer: HuggingFace tokenizer to use.
    """
    encoded = tokenizer(text)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])

    print(f"Input: \"{text}\"")
    print(f"{'Token':<20} {'ID':<8} {'Decoded'}")
    print("-" * 45)
    for token, token_id in zip(tokens, encoded['input_ids']):
        decoded = tokenizer.decode([token_id])
        print(f"{token!r:<20} {token_id:<8} {decoded!r}")


print("=== DETAILED TOKENIZATION ===")
inspect_tokenization("Machine learning is transforming healthcare.", gpt2_tokenizer)

## Part 5: Token Embeddings

Once text is tokenized into integer IDs, the model converts each ID into a dense vector through an **embedding layer**. This is a learned lookup table where each token ID maps to a vector of fixed dimension (e.g., 768 for BERT-base). These vectors are the actual inputs to the transformer layers.

In [ ]:
# Load a model and inspect its embedding layer
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

embedding_layer = model.embeddings.word_embeddings

print("=== BERT EMBEDDING LAYER ===")
print(f"Vocabulary size:    {embedding_layer.num_embeddings:,}")
print(f"Embedding dimension: {embedding_layer.embedding_dim}")
print(f"Total parameters:   {embedding_layer.num_embeddings * embedding_layer.embedding_dim:,}")
print(f"Memory:             {embedding_layer.num_embeddings * embedding_layer.embedding_dim * 4 / 1e6:.1f} MB (FP32)")

In [ ]:
# Look up embeddings for specific tokens
sample_tokens = ["the", "cat", "dog", "king", "queen", "[CLS]", "[SEP]"]

print("=== TOKEN EMBEDDINGS ===")
print(f"{'Token':<10} {'ID':<8} {'Embedding (first 8 dims)'}")
print("-" * 60)

for token in sample_tokens:
    token_id = bert_tokenizer.convert_tokens_to_ids(token)
    embedding = embedding_layer.weight[token_id].detach()
    first_dims = ", ".join(f"{x:.3f}" for x in embedding[:8])
    print(f"{token:<10} {token_id:<8} [{first_dims}, ...]")

print(f"\nEach token is a {embedding_layer.embedding_dim}-dimensional vector.")

### Embedding Similarity

Since embeddings are vectors, we can measure similarity between tokens using cosine similarity. Semantically related words should have higher similarity scores. Let's verify this with some classic examples.

In [ ]:
# Measure cosine similarity between token embeddings
from torch.nn.functional import cosine_similarity


def token_similarity(token_a: str, token_b: str) -> float:
    """Compute cosine similarity between two token embeddings.

    Args:
        token_a: First token string.
        token_b: Second token string.

    Returns:
        Cosine similarity score between -1 and 1.
    """
    id_a = bert_tokenizer.convert_tokens_to_ids(token_a)
    id_b = bert_tokenizer.convert_tokens_to_ids(token_b)
    emb_a = embedding_layer.weight[id_a].detach().unsqueeze(0)
    emb_b = embedding_layer.weight[id_b].detach().unsqueeze(0)
    return cosine_similarity(emb_a, emb_b).item()


# Test pairs
pairs = [
    ("king", "queen", "Related (royalty)"),
    ("cat", "dog", "Related (animals)"),
    ("cat", "car", "Unrelated"),
    ("happy", "sad", "Opposite"),
    ("run", "running", "Morphological"),
    ("the", "king", "Unrelated"),
]

print("=== TOKEN EMBEDDING SIMILARITY ===")
results = []
for token_a, token_b, relationship in pairs:
    sim = token_similarity(token_a, token_b)
    results.append({"Token A": token_a, "Token B": token_b, "Similarity": f"{sim:.4f}", "Relationship": relationship})

print(pd.DataFrame(results).to_string(index=False))

print("\nNote: These are *static* embeddings (before transformer layers).")
print("After passing through the transformer, context makes them much richer.")

## Part 6: Positional Encoding

Transformers process all tokens in parallel (unlike RNNs which process sequentially). This means the model has no inherent sense of word order — "the cat sat on the mat" and "mat the on sat cat the" would produce identical embeddings without positional information.

**Positional encodings** solve this by adding position-dependent information to each token embedding. BERT uses **learned** position embeddings (a separate embedding table for positions 0, 1, 2, ..., 511).

In [ ]:
# Inspect BERT's positional embeddings
pos_embeddings = model.embeddings.position_embeddings

print("=== POSITIONAL EMBEDDINGS ===")
print(f"Max positions: {pos_embeddings.num_embeddings}")
print(f"Dimension: {pos_embeddings.embedding_dim}")

# Show how positions 0, 1, 2 have different vectors
print(f"\nPosition 0 (first 6 dims): {pos_embeddings.weight[0][:6].detach().tolist()}")
print(f"Position 1 (first 6 dims): {pos_embeddings.weight[1][:6].detach().tolist()}")
print(f"Position 2 (first 6 dims): {pos_embeddings.weight[2][:6].detach().tolist()}")

# Similarity between adjacent vs distant positions
pos_0 = pos_embeddings.weight[0].detach().unsqueeze(0)
pos_1 = pos_embeddings.weight[1].detach().unsqueeze(0)
pos_100 = pos_embeddings.weight[100].detach().unsqueeze(0)
pos_511 = pos_embeddings.weight[511].detach().unsqueeze(0)

print(f"\nPosition similarity:")
print(f"  pos(0) vs pos(1):   {cosine_similarity(pos_0, pos_1).item():.4f} (adjacent)")
print(f"  pos(0) vs pos(100): {cosine_similarity(pos_0, pos_100).item():.4f} (distant)")
print(f"  pos(0) vs pos(511): {cosine_similarity(pos_0, pos_511).item():.4f} (very distant)")
print("\nAdjacent positions are more similar, as expected.")

In [ ]:
# Visualize: how token + position embeddings combine
import matplotlib.pyplot as plt

text = "The cat sat on the mat"
encoded = bert_tokenizer(text, return_tensors="pt")

with torch.no_grad():
    # Get the three embedding components
    token_embs = model.embeddings.word_embeddings(encoded['input_ids'])
    position_ids = torch.arange(encoded['input_ids'].shape[1]).unsqueeze(0)
    pos_embs = model.embeddings.position_embeddings(position_ids)

    # Combined embedding (what the transformer actually receives)
    combined = model.embeddings(encoded['input_ids'])

tokens = bert_tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Token embeddings
axes[0].imshow(token_embs[0].detach().numpy()[:, :50], aspect='auto', cmap='coolwarm')
axes[0].set_yticks(range(len(tokens)))
axes[0].set_yticklabels(tokens)
axes[0].set_title("Token Embeddings")
axes[0].set_xlabel("Dimension (first 50)")

# Position embeddings
axes[1].imshow(pos_embs[0].detach().numpy()[:len(tokens), :50], aspect='auto', cmap='coolwarm')
axes[1].set_yticks(range(len(tokens)))
axes[1].set_yticklabels([f"pos {i}" for i in range(len(tokens))])
axes[1].set_title("Position Embeddings")
axes[1].set_xlabel("Dimension (first 50)")

# Combined
axes[2].imshow(combined[0].detach().numpy()[:, :50], aspect='auto', cmap='coolwarm')
axes[2].set_yticks(range(len(tokens)))
axes[2].set_yticklabels(tokens)
axes[2].set_title("Combined (Token + Position)")
axes[2].set_xlabel("Dimension (first 50)")

plt.tight_layout()
plt.show()

print("Token embeddings + position embeddings = what the transformer actually sees.")

## Part 7: Practical Considerations

Understanding tokenization has real practical implications: context window limits, cost calculations, and input formatting all depend on token counts. Let's explore some scenarios you'll encounter when using these models.

In [ ]:
# Context window: how many tokens fit?
long_text = "This is a sample sentence. " * 200  # ~1000 words

models_and_limits = [
    ("bert-base-uncased", bert_tokenizer, 512),
    ("gpt2", gpt2_tokenizer, 1024),
]

print("=== CONTEXT WINDOW ANALYSIS ===")
print(f"Input: ~{len(long_text.split())} words\n")

for model_name, tok, max_len in models_and_limits:
    token_count = len(tok.encode(long_text))
    fits = token_count <= max_len
    print(f"{model_name}:")
    print(f"  Token count: {token_count}")
    print(f"  Max context: {max_len}")
    print(f"  Fits: {'Yes' if fits else f'No (overflow by {token_count - max_len} tokens)'}")
    print()

In [ ]:
# Truncation and padding
texts = [
    "Short.",
    "This is a medium-length sentence about tokenization.",
    "This is a longer sentence that contains more words and will require more tokens to represent in the model's vocabulary.",
]

# Without padding/truncation
print("=== WITHOUT PADDING ===")
for text in texts:
    ids = bert_tokenizer.encode(text)
    print(f"  '{text[:40]}{'...' if len(text) > 40 else ''}' -> {len(ids)} tokens")

# With padding to same length
print("\n=== WITH PADDING (max_length=32) ===")
batch = bert_tokenizer(texts, padding='max_length', truncation=True, max_length=32, return_tensors='pt')
print(f"Input IDs shape: {batch['input_ids'].shape}")
print(f"Attention mask shape: {batch['attention_mask'].shape}")

for i, text in enumerate(texts):
    real_tokens = batch['attention_mask'][i].sum().item()
    pad_tokens = 32 - real_tokens
    print(f"  Text {i+1}: {real_tokens} real tokens + {pad_tokens} padding = 32 total")

## Exercises

1. **Language Comparison**: Tokenize the same sentence in English, Spanish, and Chinese using `bert-base-multilingual-cased`. Compare token counts. Which language uses the most tokens?

2. **Code Tokenization**: Compare how GPT-2 and BERT tokenize a Python function. Which produces fewer tokens for code?

3. **Vocabulary Search**: Find all tokens in the GPT-2 vocabulary that contain "math". How many are there? What about "science"?

4. **Embedding Arithmetic**: Compute the vector `king - man + woman` using BERT token embeddings. Is the closest token "queen"?

5. **Position Heatmap**: Visualize the full 512x768 position embedding matrix as a heatmap. What patterns do you see?

In [ ]:
# Try the exercises above to deepen your understanding of tokenization.
# For example, compare multilingual tokenization:
#   multi_tok = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
#   for text in ['Hello world', 'Hola mundo', '你好世界']:
#       tokens = multi_tok.tokenize(text)
#       print(f'{text}: {len(tokens)} tokens -> {tokens}')

## Key Takeaways

- **Subword tokenization** (BPE, WordPiece) is the standard — it balances vocabulary size with sequence length
- **Different models tokenize differently** — GPT-2 (BPE) and BERT (WordPiece) produce different token counts for the same text
- **Special tokens** ([CLS], [SEP], [PAD]) serve structural roles and vary by model family
- **Embeddings** convert token IDs into dense vectors that encode meaning
- **Positional encodings** add word-order information since transformers process tokens in parallel
- **Token count matters** for context windows, API costs, and padding/truncation decisions

## Next Steps

- **Notebook 00_02**: Transformer Architecture — how self-attention processes these embeddings
- [HuggingFace Tokenizer Summary](https://huggingface.co/docs/transformers/tokenizer_summary)
- [The Illustrated Word2Vec](https://jalammar.github.io/illustrated-word2vec/)

## Resources

- [HuggingFace Tokenizers Documentation](https://huggingface.co/docs/tokenizers/)
- [BPE Paper (Sennrich et al., 2016)](https://arxiv.org/abs/1508.07909)
- [WordPiece Paper (Schuster & Nakajima, 2012)](https://research.google/pubs/pub37842/)
- [SentencePiece](https://github.com/google/sentencepiece)